## Predictive Analytics for Public Transport Disruptions


In [1]:
import os
os.environ ['PYSPARK_PYTHON'] = 'python'

In [12]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("PublicTransportPredictiveAnalytics").getOrCreate()

In [3]:
from pyspark.sql.functions import (
    col, when, hour, dayofweek,
    to_timestamp, count
)
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier
)
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    BinaryClassificationEvaluator
)

## 3. Data Ingection 

In [4]:
timetable_df = spark.read.csv ("C:/4th Semester/Siddhartha Sir/Assignment Dataset/bodsdatacatalogue/timetables_data_catalogue.csv",
                               header = True,
                               inferSchema = True
                              )
disruption_df = spark.read.csv("C:/4th Semester/Siddhartha Sir/Assignment Dataset/bodsdatacatalogue/disruptions_data_catalogue.csv",
                              header=True,
                              inferSchema=True
                              )

In [13]:
timetable_df.printSchema()
disruption_df.printSchema()


root
 |-- XML:Service Code: string (nullable = true)
 |-- XML:Line Name: string (nullable = true)
 |-- Requires Attention: string (nullable = true)
 |-- Published Status: string (nullable = true)
 |-- OTC Status: string (nullable = true)
 |-- Scope Status: string (nullable = true)
 |-- Seasonal Status: string (nullable = true)
 |-- Timeliness Status: timestamp (nullable = true)
 |-- Organisation Name: string (nullable = true)
 |-- Data set ID: string (nullable = true)
 |-- Date OTC variation needs to be published: string (nullable = true)
 |-- Date for complete 42 day look ahead: string (nullable = true)
 |-- Date when data is over 1 year old: string (nullable = true)
 |-- Date seasonal service should be published: string (nullable = true)
 |-- Seasonal Start Date: string (nullable = true)
 |-- Seasonal End Date: string (nullable = true)
 |-- XML:Filename: string (nullable = true)
 |-- XML:Last Modified Date: string (nullable = true)
 |-- XML:National Operator Code: string (nullable = 

In [14]:
def normalize_columns(df):
    for c in df.columns:
        df = df.withColumnRenamed(
            c,
            c.lower()
             .replace(" ", "_")
             .replace(":", "")
        )
    return df

timetable_df = normalize_columns(timetable_df)
disruption_df = normalize_columns(disruption_df)



In [15]:
timetable_df.columns
disruption_df.columns



['organisation',
 'id',
 'validity_start',
 'validity_end',
 'publication_start',
 'publication_end',
 'reason',
 'planned',
 'modes_affected',
 'operators_affected',
 'services_affected',
 'stops_affected']

## 4. Data Understanding

In [6]:
timetable_df.show (5)
disruption_df.show(5)

print ("Timetable rows:", timetable_df.count())
print ("Disruption rows:", disruption_df.count())

+----------------+-------------+------------------+----------------+------------+------------+---------------+--------------------+--------------------+-----------+----------------------------------------+-----------------------------------+---------------------------------+-----------------------------------------+-------------------+-----------------+--------------------+----------------------+--------------------------+------------------+-------------------+-------------------+-------------------------------+-----------------------------+--------------------+--------------------+---------------+-----------------+-----------+------------------+------------------+-----------------------+----------------------------+--------------------+------------------+---------------+----------------+-------+----------------+---------------+------------------+-----------------+------------------------------+----------------+-------------------------+
|XML:Service Code|XML:Line Name|Requires Attenti

## 5. Data Cleaning


In [16]:
timetable_df = timetable_df.dropna(
    subset=["xmlservice_code", "xmloperating_period_start_date"]
)


disruption_df = disruption_df.dropna(
    subset=["organisation", "validity_start"]
)


## Feature Engineeering

In [17]:
from pyspark.sql.functions import to_date, year, month, when

disruption_df = disruption_df.withColumn(
    "validity_start", to_date("validity_start")
)

disruption_df = disruption_df.withColumn(
    "is_planned",
    when(disruption_df.planned == True, 1).otherwise(0)
)


## Aggregate Disruptions per Organisation

In [18]:
from pyspark.sql.functions import count, sum

disruption_summary = disruption_df.groupBy("organisation").agg(
    count("*").alias("total_disruptions"),
    sum("is_planned").alias("planned_disruptions")
)


## Join Timetable with Disruptions

In [19]:
joined_df = timetable_df.join(
    disruption_summary,
    timetable_df.organisation_name == disruption_summary.organisation,
    how="left"
)


In [20]:
joined_df = joined_df.fillna(0)


## Create Prediction Target (Classification)

In [21]:
joined_df = joined_df.withColumn(
    "high_disruption_risk",
    when(joined_df.total_disruptions > 10, 1).otherwise(0)
)


## Feature Selection for ML

In [22]:
from pyspark.ml.feature import VectorAssembler

features = [
    "total_disruptions",
    "planned_disruptions"
]

assembler = VectorAssembler(
    inputCols=features,
    outputCol="features"
)

ml_df = assembler.transform(joined_df).select(
    "features", "high_disruption_risk"
)


## Train/Test Split

In [23]:
train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)


## Logistic Regression Model

In [25]:
from pyspark.sql.functions import (
    col,
    to_timestamp,
    regexp_replace
)



In [26]:
# Remove the timezone text that Spark cannot parse
disruption_df = disruption_df.withColumn(
    "validity_start_clean",
    regexp_replace(
        col("validity_start"),
        " GMT.*",
        ""
    )
)

# Convert to timestamp using correct format
disruption_df = disruption_df.withColumn(
    "validity_start_ts",
    to_timestamp(
        col("validity_start_clean"),
        "EEE MMM dd yyyy HH:mm:ss"
    )
)


In [27]:
disruption_df = disruption_df.dropna(subset=["validity_start_ts"])


In [28]:
disruption_df = disruption_df.drop("validity_start", "validity_start_clean")



In [30]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")



In [31]:
from pyspark.sql.functions import to_timestamp, col

disruption_df = disruption_df.withColumn(
    "validity_start_ts",
    to_timestamp(
        col("validity_start"),
        "EEE MMM dd yyyy HH:mm:ss 'GMT'Z"
    )
)


{"ts": "2026-01-06 14:18:43.597", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `validity_start` cannot be resolved. Did you mean one of the following? [`validity_start_ts`, `validity_end`, `publication_start`, `id`, `is_planned`]. SQLSTATE: 42703", "context": {"file": "line 6 in cell [31]", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o281.withColumn.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `validity_start` cannot be resolved. Did you mean one of the following? [`validity_start_ts`, `validity_end`, `publication_start`, `id`, `is_planned`]. SQLSTATE: 42703;\n'Project [organisation#435, id#436, validity_end#438, publication_start#439, publication_end#440, reason#441, planned

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `validity_start` cannot be resolved. Did you mean one of the following? [`validity_start_ts`, `validity_end`, `publication_start`, `id`, `is_planned`]. SQLSTATE: 42703;
'Project [organisation#435, id#436, validity_end#438, publication_start#439, publication_end#440, reason#441, planned#442, modes_affected#443, operators_affected#444, services_affected#445, stops_affected#446, is_planned#448, 'to_timestamp('validity_start, EEE MMM dd yyyy HH:mm:ss 'GMT'Z) AS validity_start_ts#502]
+- Project [organisation#435, id#436, validity_end#438, publication_start#439, publication_end#440, reason#441, planned#442, modes_affected#443, operators_affected#444, services_affected#445, stops_affected#446, is_planned#448, validity_start_ts#497]
   +- Filter atleastnnonnulls(1, validity_start_ts#497)
      +- Project [organisation#435, id#436, validity_start#447, validity_end#438, publication_start#4